# Simulated Annealing Repeat Reliability

This companion notebook shows the repeat-reliability analysis from the Simulated Annealing tutorial as a focused, reusable example. It uses the bundled `results/all_raw_runs.pkl` data and the same best/random performance-ratio convention to define success.

Load a small set of real Simulated Annealing run rows, then compute the per-instance best and random baselines used to turn a performance-ratio target into an energy threshold.

In [ ]:
from pathlib import Path

import dimod
import numpy as np
import pandas as pd

from reliability_analysis import (
    DEFAULT_QUALITY_TARGET,
    attach_quality_thresholds,
    load_selected_raw_runs,
)

example_dir = Path.cwd()
instance_ids = [0, 1, 2]
quality_target = DEFAULT_QUALITY_TARGET
num_reads = 1000
num_variables = 100


def random_energy_for_instance(instance_id, *, num_reads=1000, num_variables=100):
    np.random.seed(instance_id)
    J = np.random.rand(num_variables, num_variables)
    J = np.triu(J, 1)
    h = np.random.rand(num_variables)
    model_random = dimod.BinaryQuadraticModel.from_ising(h, J, offset=0.0)
    samples = dimod.RandomSampler().sample(
        model_random,
        num_reads=num_reads,
        seed=instance_id,
    )
    return float(np.asarray(samples.data_vectors["energy"]).mean())


runs = load_selected_raw_runs(example_dir, instance_ids)
instance_baselines = {}
for instance_id in instance_ids:
    instance_runs = runs[runs["instance"] == instance_id]
    instance_baselines[instance_id] = {
        "best_value": float(instance_runs["energy"].min()),
        "random_value": random_energy_for_instance(
            instance_id,
            num_reads=num_reads,
            num_variables=num_variables,
        ),
    }

baseline_table = (
    pd.DataFrame.from_dict(instance_baselines, orient="index")
    .rename_axis("instance")
    .reset_index()
)
baseline_table

Attach the derived energy threshold, run the public `stochastic_benchmark.run_RepeatReliability` workflow through the helper, and display compact diagnostics for selected resources.

In [ ]:
import stochastic_benchmark as SB

from reliability_analysis import (
    run_reliability_analysis,
    select_reliability_diagnostics,
)

reliability_runs = attach_quality_thresholds(
    runs,
    instance_baselines,
    quality_target=quality_target,
)

benchmark = SB.stochastic_benchmark(
    parameter_names=["sweeps"],
    here=str(example_dir),
    instance_cols=["instance"],
    response_key="energy",
    response_dir=-1,
    recover=False,
    smooth=False,
)

report = run_reliability_analysis(
    benchmark,
    reliability_runs,
    quality_target=quality_target,
    relative_error_threshold=0.10,
)
diagnostics = select_reliability_diagnostics(report)

assert not diagnostics.empty
assert report["reliability_status"].eq("needs_more_trials").any()
assert report["statistically_unresolved"].any()

diagnostics

Summarize the full report to confirm the example includes reliable, under-sampled, and statistically unresolved cases.

In [ ]:
status_summary = (
    report["reliability_status"]
    .value_counts()
    .rename_axis("reliability_status")
    .reset_index(name="rows")
)
unresolved_rows = int(report["statistically_unresolved"].sum())
print(f"Rows with statistically unresolved intervals: {unresolved_rows}")
status_summary